# AWS mushroom training

Local trials requested by Tomislav. New mushroom review assigned to Muneeb; review pending. AI assistance: [record](../docs/AI_USE.md).

Status: prepared, not run on AWS. Add the real operator and date after the run.

## Set the operator and a new run folder

 Enter your own name. Every run has its own folder so earlier evidence remains intact.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the review project.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from datetime import datetime, timezone
import subprocess, json
OPERATOR = 'YOUR NAME'
RUN = ROOT / 'reports/aws' / ('mushrooms-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
print(RUN)

## Train and tune

 The runner checks redacted SageMaker metadata. It fits preparation inside each training fold.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/train_review_aws.py'), '--dataset', 'mushrooms', '--operator', OPERATOR, '--output', str(RUN)], cwd=ROOT, check=True)

## Read the actual result

 Keep warnings and scores. Compare the same training protocol, not a summer score with a CV score.

In [ ]:
record = json.loads((RUN / 'run_record.json').read_text())
display(record)
display(pd.read_csv(RUN / 'tuning_results.csv'))
assert record['execution'] == 'AWS notebook instance'
assert record['status'] == 'complete'

## Save and export

 Save the notebook first. Then run the export cell.

In [ ]:
NOTEBOOK = ROOT / 'mushrooms/10_aws_extra_trees.ipynb'
BUNDLE = ROOT / 'mushrooms_aws_evidence.zip'
subprocess.run([sys.executable, str(ROOT / 'scripts/export_aws_evidence.py'),
                '--notebook', str(NOTEBOOK), '--run', str(RUN),
                '--output', str(BUNDLE)], cwd=ROOT, check=True)
print(BUNDLE.name)